# BME Deep Learning — Lab 02 (Guided)
## PyTorch Fundamentals and Convolutional Neural Networks

**Course:** Deep Learning / VITMMA19  
**Budapest University of Technology and Economics (BME)**  
**Department of Telecommunications and Artificial Intelligence**  
**Academic year:** 2026/2027 — Fall semester
**Instructor: Dr. Mohammed Salah Al-Radhi

This is the **guided part** of Lab 02. We will build the complete workflow together before you start the independent assignment.

### Today you will learn to
- work with PyTorch tensors and move data/models between CPU and GPU,
- use automatic differentiation with `autograd`,
- load image datasets using `Dataset` and `DataLoader`,
- understand the basic building blocks of a CNN,
- train a CNN with a standard PyTorch training loop,
- evaluate predictions using accuracy and a confusion matrix.

> **Connection to Lab 01:** Last week you worked with tensors, gradients, loss functions and parameter updates. Today we use the same ideas inside PyTorch and apply them to image classification.

## 0. Colab setup

For this guided notebook, a CPU is sufficient, but a GPU will make training faster.

In Google Colab you may select:
**Runtime → Change runtime type → T4 GPU**.

Run the cells in order. The notebook is intentionally written with the full PyTorch workflow visible — no high-level training framework is used.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch version:', torch.__version__)
print('Device:', DEVICE)

## 1. PyTorch tensors — a short refresher

A PyTorch tensor is similar to a NumPy array, but it can:
1. run efficiently on accelerators such as GPUs, and
2. track operations so gradients can be computed automatically.

In [ ]:
# Create tensors in a few common ways
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y = torch.randn(2, 2)
z = x @ y

print('x =\n', x)
print('y =\n', y)
print('x @ y =\n', z)
print('shape:', z.shape)
print('dtype:', z.dtype)
print('device:', z.device)

In [ ]:
# Moving a tensor to the selected device
x_device = x.to(DEVICE)
print('Tensor device:', x_device.device)

### Why device handling matters

The **model and its input batch must be on the same device**. A common PyTorch error comes from putting the model on the GPU while leaving the data on the CPU (or vice versa).

## 2. Automatic differentiation (`autograd`)

In Lab 01, you saw the idea of computing a loss and then updating parameters using a gradient. PyTorch records the operations that create a tensor when `requires_grad=True`.

In [ ]:
# Simple scalar example: L = (w - 3)^2
w = torch.tensor(0.0, requires_grad=True)
loss = (w - 3.0) ** 2
loss.backward()

print('w:', w.item())
print('loss:', loss.item())
print('dL/dw:', w.grad.item())

The derivative of $(w-3)^2$ is $2(w-3)$. At `w = 0`, the gradient is `-6`, exactly what PyTorch computed.

During neural-network training, `loss.backward()` performs the same idea for **all trainable model parameters**.

## 3. Dataset and DataLoader

We will use **Fashion-MNIST** for the guided example. It contains 28×28 grayscale images from 10 clothing categories.

The standard PyTorch pipeline is:

`Dataset → transform → DataLoader → mini-batches`

In [ ]:
# Convert images to tensors and normalize pixels approximately to [-1, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_train = datasets.FashionMNIST(root='data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='data', train=False, download=True, transform=transform)

# Use a moderate subset so the guided training finishes quickly in class.
train_dataset, val_dataset, _ = random_split(
    full_train,
    [24000, 6000, 30000],
    generator=torch.Generator().manual_seed(SEED)
)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                          pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                        pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                         pin_memory=torch.cuda.is_available())

print('Training samples:', len(train_dataset))
print('Validation samples:', len(val_dataset))
print('Test samples:', len(test_dataset))

In [ ]:
images, labels = next(iter(train_loader))
print('Image batch shape:', images.shape)
print('Label batch shape:', labels.shape)
print('Pixel range after normalization:', float(images.min()), 'to', float(images.max()))

### Reading the batch shape

`[128, 1, 28, 28]` means:
- `128` images in this mini-batch,
- `1` input channel (grayscale),
- image height `28`,
- image width `28`.

PyTorch image tensors normally use the order **N × C × H × W**.

In [ ]:
class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    # Undo normalization for display: x_original = x_norm * 0.5 + 0.5
    img = images[i].squeeze().cpu() * 0.5 + 0.5
    ax.imshow(img, cmap='gray')
    ax.set_title(class_names[labels[i].item()])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. What does a CNN do?

A convolutional neural network learns **local visual patterns** using small filters.

A typical CNN contains:
- **Convolution (`Conv2d`)** — learns local filters such as edges, textures and shapes,
- **ReLU** — introduces non-linearity,
- **Pooling (`MaxPool2d`)** — reduces spatial resolution,
- **Flatten + Linear** — converts learned feature maps into class scores.

The final outputs are called **logits**. We do **not** manually apply softmax when using `nn.CrossEntropyLoss`, because that loss handles the required operation internally.

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 28x28 -> 28x28
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 28x28 -> 14x14
            nn.Conv2d(16, 32, kernel_size=3, padding=1), # 14x14 -> 14x14
            nn.ReLU(),
            nn.MaxPool2d(2)                               # 14x14 -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = FashionCNN().to(DEVICE)
print(model)

### Shape check before training

A very useful debugging habit is to pass a **dummy mini-batch** through the model before training. For 10-class classification, the model should return one vector of 10 logits per image.

In [ ]:
with torch.no_grad():
    dummy = torch.randn(8, 1, 28, 28).to(DEVICE)
    output = model(dummy)

print('Input shape :', dummy.shape)
print('Output shape:', output.shape)
assert output.shape == (8, 10)

## 5. Loss function and optimizer

For a multi-class classification problem with one correct label per image:
- use `nn.CrossEntropyLoss()` for the classification loss,
- use an optimizer such as `Adam` to update the model parameters.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 6. The PyTorch training loop

Every mini-batch follows the same pattern:

1. move data to the device,
2. clear old gradients,
3. compute logits (forward pass),
4. compute the loss,
5. compute gradients with `loss.backward()`,
6. update parameters with `optimizer.step()`.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += images.size(0)

    return total_loss / total_samples, total_correct / total_samples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += images.size(0)

    return total_loss / total_samples, total_correct / total_samples

### `model.train()` vs `model.eval()`

These calls matter because layers such as **Dropout** and **Batch Normalization** behave differently during training and evaluation.

Also notice `@torch.no_grad()` on evaluation: gradients are not needed there, so disabling them saves memory and computation.

In [ ]:
EPOCHS = 3
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(
        f'Epoch {epoch:02d}/{EPOCHS} | '
        f'train loss {train_loss:.4f} | train acc {train_acc:.3f} | '
        f'val loss {val_loss:.4f} | val acc {val_acc:.3f}'
    )

## 7. Learning curves

Training accuracy tells us how well the model fits examples it has seen. Validation accuracy gives a better indication of performance on unseen data.

In [ ]:
epochs = range(1, EPOCHS + 1)

plt.figure(figsize=(6, 4))
plt.plot(epochs, history['train_loss'], marker='o', label='Train loss')
plt.plot(epochs, history['val_loss'], marker='o', label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('Loss curves')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(epochs, history['train_acc'], marker='o', label='Train accuracy')
plt.plot(epochs, history['val_acc'], marker='o', label='Validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy curves')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Discussion

Look at the two curves and ask:
- Are both training and validation loss decreasing?
- Is there a large gap between training and validation accuracy?
- Would more epochs probably help, or are we beginning to overfit?

## 8. Test evaluation and confusion matrix

The test set should be used **after** model development. It gives us a final estimate on data that was not used for parameter updates or model selection.

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.3f}')

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    ys, preds = [], []
    for images, labels in loader:
        logits = model(images.to(device))
        preds.extend(logits.argmax(dim=1).cpu().numpy())
        ys.extend(labels.numpy())
    return np.array(ys), np.array(preds)

y_true, y_pred = collect_predictions(model, test_loader, DEVICE)
cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(9, 9))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title('Fashion-MNIST confusion matrix')
plt.show()

## 9. Inspect individual predictions

A single accuracy value does not explain **what kind of mistakes** the model makes. Looking at predictions helps us understand model behaviour.

In [ ]:
images, labels = next(iter(test_loader))
with torch.no_grad():
    logits = model(images.to(DEVICE))
    predictions = logits.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.flat):
    img = images[i].squeeze() * 0.5 + 0.5
    ok = predictions[i].item() == labels[i].item()
    ax.imshow(img, cmap='gray')
    ax.set_title(
        f"pred: {class_names[predictions[i]]}\ntrue: {class_names[labels[i]]}\n"
        + ('correct' if ok else 'wrong')
    )
    ax.axis('off')
plt.tight_layout()
plt.show()

# What to remember before the assignment

You should now be able to explain this pipeline:

**images → DataLoader → CNN → logits → CrossEntropyLoss → backward → optimizer step → validation**

Key debugging checks:
1. verify the input shape (`N × C × H × W`),
2. verify the output shape (`N × number_of_classes`),
3. keep model and tensors on the same device,
4. use `model.train()` during training and `model.eval()` during evaluation,
5. call `optimizer.zero_grad()` before each backward pass,
6. do not apply softmax before `CrossEntropyLoss`.

In the assignment, you will transfer these ideas from Fashion-MNIST to the more difficult **CIFAR-10** dataset.